In [1]:
!pip install -q datasets pandas scikit-learn kagglehub

In [2]:
from pathlib import Path
import glob
import hashlib

import pandas as pd
from datasets import load_dataset
import kagglehub
from sklearn.model_selection import train_test_split

OUT_DIR = Path("/content/concierge_classifier_data")
OUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

FINAL_LABELS = {
    "faq",
    "support",
    "sales_or_leads",
    "human_request",
    "spam",
    "other",
}

In [3]:
BITEXT_DATASET = "bitext/Bitext-customer-support-llm-chatbot-training-dataset"

bitext_ds = load_dataset(BITEXT_DATASET)

print(bitext_ds)
print("Splits:", list(bitext_ds.keys()))

bitext_split = list(bitext_ds.keys())[0]
bitext_df = pd.DataFrame(bitext_ds[bitext_split])

print("Bitext shape:", bitext_df.shape)
print("Bitext columns:", bitext_df.columns.tolist())

bitext_df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['flags', 'instruction', 'category', 'intent', 'response'],
        num_rows: 26872
    })
})
Splits: ['train']
Bitext shape: (26872, 5)
Bitext columns: ['flags', 'instruction', 'category', 'intent', 'response']


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [4]:
print("Categories:")
print(bitext_df["category"].value_counts())

print("\nIntents:")
print(bitext_df["intent"].value_counts())

Categories:
category
ACCOUNT         5986
ORDER           3988
REFUND          2992
CONTACT         1999
INVOICE         1999
PAYMENT         1998
FEEDBACK        1997
DELIVERY        1994
SHIPPING        1970
SUBSCRIPTION     999
CANCEL           950
Name: count, dtype: int64

Intents:
intent
contact_customer_service    1000
complaint                   1000
check_invoice               1000
switch_account              1000
edit_account                1000
contact_human_agent          999
check_payment_methods        999
delivery_period              999
newsletter_subscription      999
get_invoice                  999
payment_issue                999
registration_problems        999
cancel_order                 998
place_order                  998
track_refund                 998
change_order                 997
set_up_shipping_address      997
check_refund_policy          997
create_account               997
get_refund                   997
review                       997
delivery_opt

In [5]:
def map_bitext_to_concierge_label(row):
    intent = str(row["intent"]).lower().strip()

    intent_to_label = {
        # Human escalation
        "contact_customer_service": "human_request",
        "contact_human_agent": "human_request",

        # Sales / lead-like intent
        "create_account": "sales_or_leads",
        "place_order": "sales_or_leads",
        "newsletter_subscription": "sales_or_leads",

        # FAQ / simple informational questions
        "check_invoice": "faq",
        "get_invoice": "faq",
        "check_payment_methods": "faq",
        "delivery_period": "faq",
        "delivery_options": "faq",
        "track_order": "faq",
        "track_refund": "faq",
        "check_refund_policy": "faq",
        "check_cancellation_fee": "faq",

        # Support / problem-solving
        "complaint": "support",
        "payment_issue": "support",
        "registration_problems": "support",
        "recover_password": "support",
        "edit_account": "support",
        "switch_account": "support",
        "delete_account": "support",
        "cancel_order": "support",
        "change_order": "support",
        "get_refund": "support",
        "change_shipping_address": "support",
        "set_up_shipping_address": "support",
        "review": "support",
    }

    return intent_to_label.get(intent, "support")


bitext_clean = pd.DataFrame({
    "text": bitext_df["instruction"].astype(str),
    "label": bitext_df.apply(map_bitext_to_concierge_label, axis=1),
    "source": "bitext",
    "original_intent": bitext_df["intent"].astype(str),
    "original_category": bitext_df["category"].astype(str),
})

bitext_clean = bitext_clean.dropna(subset=["text", "label"])
bitext_clean["text"] = bitext_clean["text"].astype(str).str.strip()
bitext_clean["label"] = bitext_clean["label"].astype(str).str.strip()
bitext_clean = bitext_clean[bitext_clean["text"] != ""]

print("Bitext cleaned shape:", bitext_clean.shape)
print("\nMapped Bitext labels:")
print(bitext_clean["label"].value_counts())

print("\nOriginal intents mapped to project labels:")
print(
    bitext_clean[["original_intent", "label"]]
    .drop_duplicates()
    .sort_values(["label", "original_intent"])
    .to_string(index=False)
)

bitext_clean.head()

Bitext cleaned shape: (26872, 5)

Mapped Bitext labels:
label
support           12947
faq                8932
sales_or_leads     2994
human_request      1999
Name: count, dtype: int64

Original intents mapped to project labels:
         original_intent          label
  check_cancellation_fee            faq
           check_invoice            faq
   check_payment_methods            faq
     check_refund_policy            faq
        delivery_options            faq
         delivery_period            faq
             get_invoice            faq
             track_order            faq
            track_refund            faq
contact_customer_service  human_request
     contact_human_agent  human_request
          create_account sales_or_leads
 newsletter_subscription sales_or_leads
             place_order sales_or_leads
            cancel_order        support
            change_order        support
 change_shipping_address        support
               complaint        support
          de

,text,label,source,original_intent,original_category
0,question about cancelling order {{Order Number}},support,bitext,cancel_order,ORDER
1,i have a question about cancelling oorder {{Or...,support,bitext,cancel_order,ORDER
2,i need help cancelling puchase {{Order Number}},support,bitext,cancel_order,ORDER
3,I need to cancel purchase {{Order Number}},support,bitext,cancel_order,ORDER
4,"I cannot afford this order, cancel purchase {{...",support,bitext,cancel_order,ORDER


In [6]:
bitext_clean.to_csv(
    OUT_DIR / "bitext_mapped_to_concierge_labels.csv",
    index=False
)

print("Saved:", OUT_DIR / "bitext_mapped_to_concierge_labels.csv")

Saved: /content/concierge_classifier_data/bitext_mapped_to_concierge_labels.csv


In [7]:
SMS_KAGGLE_DATASET = "uciml/sms-spam-collection-dataset"

sms_path = kagglehub.dataset_download(SMS_KAGGLE_DATASET)

print("Downloaded to:", sms_path)

csv_files = glob.glob(str(Path(sms_path) / "*.csv"))
print("CSV files:", csv_files)

Using Colab cache for faster access to the 'sms-spam-collection-dataset' dataset.
Downloaded to: /kaggle/input/sms-spam-collection-dataset
CSV files: ['/kaggle/input/sms-spam-collection-dataset/spam.csv']


In [8]:
sms_csv_path = csv_files[0]

sms_raw = pd.read_csv(sms_csv_path, encoding="latin-1")

print("SMS raw shape:", sms_raw.shape)
print("SMS raw columns:", sms_raw.columns.tolist())

sms_raw.head()

SMS raw shape: (5572, 5)
SMS raw columns: ['v1', 'v2', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4']


,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [9]:
# Kaggle SMS Spam Collection usually has:
# v1 = ham/spam
# v2 = message

if {"v1", "v2"}.issubset(set(sms_raw.columns)):
    sms_base = sms_raw[["v1", "v2"]].copy()
    sms_base = sms_base.rename(columns={
        "v1": "sms_label",
        "v2": "text",
    })
elif {"label", "message"}.issubset(set(sms_raw.columns)):
    sms_base = sms_raw[["label", "message"]].copy()
    sms_base = sms_base.rename(columns={
        "label": "sms_label",
        "message": "text",
    })
else:
    raise ValueError(f"Unexpected SMS columns: {sms_raw.columns.tolist()}")

sms_base["sms_label"] = sms_base["sms_label"].astype(str).str.lower().str.strip()
sms_base["text"] = sms_base["text"].astype(str).str.strip()

print("Original SMS labels:")
print(sms_base["sms_label"].value_counts())

sms_base.head()

Original SMS labels:
sms_label
ham     4825
spam     747
Name: count, dtype: int64


,sms_label,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [11]:
# Cell 10 — Create only SMS spam rows

sms_spam_raw = sms_base[sms_base["sms_label"] == "spam"].copy()

sms_spam_clean = pd.DataFrame({
    "text": sms_spam_raw["text"],
    "label": "spam",
    "source": "sms_spam_collection",
    "original_intent": "spam",
    "original_category": "spam",
})

sms_spam_clean = sms_spam_clean.dropna(subset=["text", "label"])
sms_spam_clean["text"] = sms_spam_clean["text"].astype(str).str.strip()
sms_spam_clean = sms_spam_clean[sms_spam_clean["text"] != ""]

print("SMS spam shape:", sms_spam_clean.shape)
sms_spam_clean.head()

SMS spam shape: (747, 5)


,text,label,source,original_intent,original_category
2,Free entry in 2 a wkly comp to win FA Cup fina...,spam,sms_spam_collection,spam,spam
5,FreeMsg Hey there darling it's been 3 week's n...,spam,sms_spam_collection,spam,spam
8,WINNER!! As a valued network customer you have...,spam,sms_spam_collection,spam,spam
9,Had your mobile 11 months or more? U R entitle...,spam,sms_spam_collection,spam,spam
11,"SIX chances to win CASH! From 100 to 20,000 po...",spam,sms_spam_collection,spam,spam


In [12]:
# Cell 11 — Save SMS spam extracted file

sms_spam_clean.to_csv(
    OUT_DIR / "sms_spam_mapped_to_concierge_labels.csv",
    index=False
)

print("Saved SMS spam file:")
print(OUT_DIR / "sms_spam_mapped_to_concierge_labels.csv")

Saved SMS spam file:
/content/concierge_classifier_data/sms_spam_mapped_to_concierge_labels.csv


In [22]:
# Cell 12 — Load original CLINC/OOS dataset for the "other" label

from datasets import load_dataset, get_dataset_config_names
import pandas as pd

CLINC_DATASET = "clinc/clinc_oos"

print("Available configs:")
try:
    print(get_dataset_config_names(CLINC_DATASET))
except Exception as e:
    print("Could not fetch config names:", e)

# CLINC/OOS configs are usually: small, imbalanced, plus
# Use plus because it is the richer version.
CLINC_CONFIG = "plus"

clinc_ds = load_dataset(CLINC_DATASET, CLINC_CONFIG)

print(clinc_ds)
print("Splits:", list(clinc_ds.keys()))

for split_name in clinc_ds.keys():
    print("\nSPLIT:", split_name)
    print(clinc_ds[split_name].features)
    print("Rows:", len(clinc_ds[split_name]))

Available configs:


README.md: 0.00B [00:00, ?B/s]

['imbalanced', 'plus', 'small']


plus/train-00000-of-00001.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

plus/validation-00000-of-00001.parquet:   0%|          | 0.00/77.8k [00:00<?, ?B/s]

plus/test-00000-of-00001.parquet:   0%|          | 0.00/136k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15250 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3100 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'intent'],
        num_rows: 15250
    })
    validation: Dataset({
        features: ['text', 'intent'],
        num_rows: 3100
    })
    test: Dataset({
        features: ['text', 'intent'],
        num_rows: 5500
    })
})
Splits: ['train', 'validation', 'test']

SPLIT: train
{'text': Value('string'), 'intent': ClassLabel(names=['restaurant_reviews', 'nutrition_info', 'account_blocked', 'oil_change_how', 'time', 'weather', 'redeem_rewards', 'interest_rate', 'gas_type', 'accept_reservations', 'smart_home', 'user_name', 'report_lost_card', 'repeat', 'whisper_mode', 'what_are_your_hobbies', 'order', 'jump_start', 'schedule_meeting', 'meeting_schedule', 'freeze_account', 'what_song', 'meaning_of_life', 'restaurant_reservation', 'traffic', 'make_call', 'text', 'bill_balance', 'improve_credit_score', 'change_language', 'no', 'measurement_conversion', 'timer', 'flip_coin', 'do_you_have_pets', 'balance', 'tell_joke', 'last_maint

In [23]:
# Cell 13 — Inspect CLINC splits and columns

for split_name in clinc_ds.keys():
    split_df = pd.DataFrame(clinc_ds[split_name])

    print("\n==============================")
    print("Split:", split_name)
    print("Shape:", split_df.shape)
    print("Columns:", split_df.columns.tolist())
    print(split_df.head())


Split: train
Shape: (15250, 2)
Columns: ['text', 'intent']
                                                text  intent
0  what expression would i use to say i love you ...      61
1  can you tell me how to say 'i do not speak muc...      61
2  what is the equivalent of, 'life is good' in f...      61
3  tell me how to say, 'it is a beautiful morning...      61
4  if i were mongolian, how would i say that i am...      61

Split: validation
Shape: (3100, 2)
Columns: ['text', 'intent']
                                        text  intent
0   in spanish, meet me tomorrow is said how      61
1     in french, how do i say, see you later      61
2           how do you say hello in japanese      61
3  how do i ask about the weather in chinese      61
4  how can i say "cancel my order" in french      61

Split: test
Shape: (5500, 2)
Columns: ['text', 'intent']
                                  text  intent
0     how would you say fly in italian      61
1    what's the spanish word for pasta  

In [24]:
# Cell 13B — Convert CLINC numeric intent IDs to intent names

features = clinc_ds["train"].features
print("Features:")
print(features)

intent_feature = features["intent"]

print("\nIntent feature:")
print(intent_feature)

# Print all intent names if available
if hasattr(intent_feature, "names"):
    print("\nNumber of intent names:", len(intent_feature.names))
    print("\nIntent names:")
    for i, name in enumerate(intent_feature.names):
        print(i, "->", name)
else:
    print("No intent names found in the feature.")

Features:
{'text': Value('string'), 'intent': ClassLabel(names=['restaurant_reviews', 'nutrition_info', 'account_blocked', 'oil_change_how', 'time', 'weather', 'redeem_rewards', 'interest_rate', 'gas_type', 'accept_reservations', 'smart_home', 'user_name', 'report_lost_card', 'repeat', 'whisper_mode', 'what_are_your_hobbies', 'order', 'jump_start', 'schedule_meeting', 'meeting_schedule', 'freeze_account', 'what_song', 'meaning_of_life', 'restaurant_reservation', 'traffic', 'make_call', 'text', 'bill_balance', 'improve_credit_score', 'change_language', 'no', 'measurement_conversion', 'timer', 'flip_coin', 'do_you_have_pets', 'balance', 'tell_joke', 'last_maintenance', 'exchange_rate', 'uber', 'car_rental', 'credit_limit', 'oos', 'shopping_list', 'expiration_date', 'routing', 'meal_suggestion', 'tire_change', 'todo_list', 'card_declined', 'rewards_balance', 'change_accent', 'vaccines', 'reminder_update', 'food_last', 'change_ai_name', 'bill_due', 'who_do_you_work_for', 'share_location', 

In [25]:
# Cell 13C — Build CLINC dataframe with readable intent names

clinc_parts = []

intent_feature = clinc_ds["train"].features["intent"]

for split_name in clinc_ds.keys():
    split_df = pd.DataFrame(clinc_ds[split_name])
    split_df["split"] = split_name

    # Convert numeric intent IDs to names
    split_df["intent_name"] = split_df["intent"].apply(
        lambda x: intent_feature.int2str(int(x))
    )

    clinc_parts.append(split_df)

clinc_df = pd.concat(clinc_parts, ignore_index=True)

print("CLINC shape:", clinc_df.shape)
print(clinc_df.head())

print("\nIntent name counts:")
print(clinc_df["intent_name"].value_counts().head(50))

print("\nIntent names containing oos/out:")
print([
    name for name in sorted(clinc_df["intent_name"].unique())
    if "oos" in name.lower() or "out" in name.lower()
])

CLINC shape: (23850, 4)
                                                text  intent  split  \
0  what expression would i use to say i love you ...      61  train   
1  can you tell me how to say 'i do not speak muc...      61  train   
2  what is the equivalent of, 'life is good' in f...      61  train   
3  tell me how to say, 'it is a beautiful morning...      61  train   
4  if i were mongolian, how would i say that i am...      61  train   

  intent_name  
0   translate  
1   translate  
2   translate  
3   translate  
4   translate  

Intent name counts:
intent_name
oos                          1350
transfer                      150
translate                     150
definition                    150
meaning_of_life               150
insurance_change              150
timer                         150
travel_alert                  150
pto_request                   150
improve_credit_score          150
fun_fact                      150
change_language               150
payday      

In [26]:
# Cell 14 — Extract only explicit CLINC OOS rows as "other"

intent_feature = clinc_ds["train"].features["intent"]

clinc_parts = []

for split_name in clinc_ds.keys():
    split_df = pd.DataFrame(clinc_ds[split_name]).copy()
    split_df["split"] = split_name

    # Convert numeric intent ID to readable name
    split_df["intent_name"] = split_df["intent"].apply(
        lambda x: intent_feature.int2str(int(x))
    )

    clinc_parts.append(split_df)

clinc_df = pd.concat(clinc_parts, ignore_index=True)

clinc_df["intent_name_clean"] = (
    clinc_df["intent_name"]
    .astype(str)
    .str.lower()
    .str.strip()
)

print("CLINC intent counts containing oos:")
print(clinc_df[clinc_df["intent_name_clean"] == "oos"]["split"].value_counts())

# Take only explicit OOS rows
other_raw = clinc_df[clinc_df["intent_name_clean"] == "oos"].copy()

if other_raw.empty:
    raise ValueError("No CLINC rows with intent_name == 'oos' found.")

clinc_other_clean = pd.DataFrame({
    "text": other_raw["text"].astype(str),
    "label": "other",
    "source": "clinc_oos_plus",
    "original_intent": other_raw["intent_name"].astype(str),
    "original_category": "out_of_scope",
})

clinc_other_clean = clinc_other_clean.dropna(subset=["text", "label"])
clinc_other_clean["text"] = clinc_other_clean["text"].astype(str).str.strip()
clinc_other_clean = clinc_other_clean[clinc_other_clean["text"] != ""]

print("CLINC other shape:", clinc_other_clean.shape)

print("\nOriginal intents used for other:")
print(clinc_other_clean["original_intent"].value_counts())

print("\nSample other examples:")
display(clinc_other_clean.sample(min(10, len(clinc_other_clean)), random_state=42))

clinc_other_clean.head()

CLINC intent counts containing oos:
split
test          1000
train          250
validation     100
Name: count, dtype: int64
CLINC other shape: (1350, 5)

Original intents used for other:
original_intent
oos    1350
Name: count, dtype: int64

Sample other examples:


,text,label,source,original_intent,original_category
18289,what university in the united states offers th...,other,clinc_oos_plus,oos,out_of_scope
23536,find out how i can best clip my dog's nails,other,clinc_oos_plus,oos,out_of_scope
23035,what are the last 4 digits of my credit card n...,other,clinc_oos_plus,oos,out_of_scope
18346,what site publishes the most fake news,other,clinc_oos_plus,oos,out_of_scope
23575,update my status on instagram,other,clinc_oos_plus,oos,out_of_scope
23282,what movies are playing,other,clinc_oos_plus,oos,out_of_scope
18303,is marijuana addictive,other,clinc_oos_plus,oos,out_of_scope
23254,tell me who aphrodite is,other,clinc_oos_plus,oos,out_of_scope
23036,switch over to low power mode to preserve batt...,other,clinc_oos_plus,oos,out_of_scope
15076,where can i see art,other,clinc_oos_plus,oos,out_of_scope


,text,label,source,original_intent,original_category
15000,how much is an overdraft fee for bank,other,clinc_oos_plus,oos,out_of_scope
15001,why are exponents preformed before multiplicat...,other,clinc_oos_plus,oos,out_of_scope
15002,what size wipers does this car take,other,clinc_oos_plus,oos,out_of_scope
15003,where is the dipstick,other,clinc_oos_plus,oos,out_of_scope
15004,how much is 1 share of aapl,other,clinc_oos_plus,oos,out_of_scope


In [27]:
# Safety check: only CLINC OOS rows are used as other

used_intents = set(
    clinc_other_clean["original_intent"]
    .astype(str)
    .str.lower()
    .str.strip()
    .unique()
)

assert used_intents == {"oos"}, f"Non-OOS intents found: {used_intents}"

assert set(clinc_other_clean["label"].unique()) == {"other"}

print("Good: only CLINC rows with original_intent = 'oos' are mapped to other.")

Good: only CLINC rows with original_intent = 'oos' are mapped to other.


In [28]:
# Cell 15 — Save CLINC other extracted file

clinc_other_clean.to_csv(
    OUT_DIR / "clinc_oos_mapped_to_other.csv",
    index=False
)

print("Saved CLINC other file:")
print(OUT_DIR / "clinc_oos_mapped_to_other.csv")

Saved CLINC other file:
/content/concierge_classifier_data/clinc_oos_mapped_to_other.csv


In [32]:
# Cell 16 — Combine Bitext + SMS spam + CLINC other

# Safety checks before combining
assert set(sms_spam_clean["label"].unique()) == {"spam"}, "SMS dataset should only contain spam."

assert set(clinc_other_clean["label"].unique()) == {"other"}, "CLINC dataset should only contain other."

assert set(
    clinc_other_clean["original_intent"].astype(str).str.lower().str.strip().unique()
) == {"oos"}, "CLINC other must come only from original_intent = oos."

# Optional but recommended:
# Bitext should not contain spam or other if you are using CLINC-only for other
allowed_bitext_labels = {
    "faq",
    "support",
    "sales_or_leads",
    "human_request",
}

bitext_labels = set(bitext_clean["label"].unique())

assert bitext_labels.issubset(allowed_bitext_labels), (
    f"Unexpected Bitext labels found: {bitext_labels - allowed_bitext_labels}"
)

combined_df = pd.concat(
    [
        bitext_clean,
        sms_spam_clean,
        clinc_other_clean,
    ],
    ignore_index=True,
)

combined_df = combined_df.dropna(subset=["text", "label"])
combined_df["text"] = combined_df["text"].astype(str).str.strip()
combined_df["label"] = combined_df["label"].astype(str).str.strip()
combined_df = combined_df[combined_df["text"] != ""]

# Remove exact duplicate text-label pairs
combined_df = combined_df.drop_duplicates(subset=["text", "label"])

# Shuffle
combined_df = combined_df.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print("Combined dataset shape:", combined_df.shape)

print("\nFinal label counts:")
print(combined_df["label"].value_counts())

print("\nUnique labels:")
print(sorted(combined_df["label"].unique()))

# Safety check
unexpected_labels = set(combined_df["label"].unique()) - FINAL_LABELS
missing_labels = FINAL_LABELS - set(combined_df["label"].unique())

print("\nUnexpected labels:", unexpected_labels)
print("Missing labels:", missing_labels)

assert not unexpected_labels, f"Unexpected labels found: {unexpected_labels}"
assert not missing_labels, f"Missing labels: {missing_labels}"

combined_df.head()

Combined dataset shape: (26627, 5)

Final label counts:
label
support           11798
faq                7949
sales_or_leads     2889
human_request      1999
other              1350
spam                642
Name: count, dtype: int64

Unique labels:
['faq', 'human_request', 'other', 'sales_or_leads', 'spam', 'support']

Unexpected labels: set()
Missing labels: set()


,text,label,source,original_intent,original_category
0,i need information about the cancellation of a...,support,bitext,delete_account,ACCOUNT
1,I need assistance to purchase some of your art...,sales_or_leads,bitext,place_order,ORDER
2,I'd like to order some of ur article how coupd...,sales_or_leads,bitext,place_order,ORDER
3,i want assistance to notify of a signup error,support,bitext,registration_problems,ACCOUNT
4,can I check what hours customer assistance ava...,human_request,bitext,contact_customer_service,CONTACT


In [33]:
# Cell 17 — Save full combined dataset

combined_df.to_csv(
    OUT_DIR / "concierge_intent_spam_other_full_debug.csv",
    index=False
)

combined_df[["text", "label"]].to_csv(
    OUT_DIR / "concierge_intent_spam_other_trainable_full.csv",
    index=False
)

print("Saved full combined dataset:")
print(OUT_DIR / "concierge_intent_spam_other_full_debug.csv")
print(OUT_DIR / "concierge_intent_spam_other_trainable_full.csv")

Saved full combined dataset:
/content/concierge_classifier_data/concierge_intent_spam_other_full_debug.csv
/content/concierge_classifier_data/concierge_intent_spam_other_trainable_full.csv


In [34]:
# Cell 18 — Create capped dataset

MAX_PER_LABEL = 3000

sampled_parts = []

for label, group in combined_df.groupby("label"):
    n_samples = min(len(group), MAX_PER_LABEL)

    sampled_group = group.sample(
        n=n_samples,
        random_state=RANDOM_STATE
    )

    sampled_parts.append(sampled_group)

capped_df = (
    pd.concat(sampled_parts, ignore_index=True)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

print("Capped shape:", capped_df.shape)

print("\nCapped label counts:")
print(capped_df["label"].value_counts())

capped_df.head()

Capped shape: (12880, 5)

Capped label counts:
label
support           3000
faq               3000
sales_or_leads    2889
human_request     1999
other             1350
spam               642
Name: count, dtype: int64


,text,label,source,original_intent,original_category
0,I want help talking with a person,human_request,bitext,contact_human_agent,CONTACT
1,help me submit some feedback about a product,support,bitext,review,FEEDBACK
2,assistance seeing in what situations can i ask...,faq,bitext,check_refund_policy,REFUND
3,how to plan for weddings,other,clinc_oos_plus,oos,out_of_scope
4,help me to shop some items,sales_or_leads,bitext,place_order,ORDER


In [35]:
# Cell 19 — Create trainable_df with only text and label

# Save full capped version for debugging/model card
capped_df.to_csv(
    OUT_DIR / "concierge_intent_spam_other_capped_full_debug.csv",
    index=False
)

# Actual training dataset: only text and label
trainable_df = capped_df[["text", "label"]].copy()

trainable_df = trainable_df.dropna(subset=["text", "label"])
trainable_df["text"] = trainable_df["text"].astype(str).str.strip()
trainable_df["label"] = trainable_df["label"].astype(str).str.strip()
trainable_df = trainable_df[trainable_df["text"] != ""]

print("Trainable shape:", trainable_df.shape)

print("\nTrainable label counts:")
print(trainable_df["label"].value_counts())

print("\nUnique trainable labels:")
print(sorted(trainable_df["label"].unique()))

trainable_df.head()

Trainable shape: (12880, 2)

Trainable label counts:
label
support           3000
faq               3000
sales_or_leads    2889
human_request     1999
other             1350
spam               642
Name: count, dtype: int64

Unique trainable labels:
['faq', 'human_request', 'other', 'sales_or_leads', 'spam', 'support']


,text,label
0,I want help talking with a person,human_request
1,help me submit some feedback about a product,support
2,assistance seeing in what situations can i ask...,faq
3,how to plan for weddings,other
4,help me to shop some items,sales_or_leads


In [36]:
# Cell 20 — Split into train, validation, and test

train_df, temp_df = train_test_split(
    trainable_df,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=trainable_df["label"],
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_df["label"],
)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nVal labels:")
print(val_df["label"].value_counts())

print("\nTest labels:")
print(test_df["label"].value_counts())

Train: (9016, 2)
Val: (1932, 2)
Test: (1932, 2)

Train labels:
label
faq               2100
support           2100
sales_or_leads    2022
human_request     1399
other              945
spam               450
Name: count, dtype: int64

Val labels:
label
support           450
faq               450
sales_or_leads    434
human_request     300
other             202
spam               96
Name: count, dtype: int64

Test labels:
label
support           450
faq               450
sales_or_leads    433
human_request     300
other             203
spam               96
Name: count, dtype: int64


In [37]:
# Cell 21 — Save final train/val/test files

trainable_df.to_csv(
    OUT_DIR / "concierge_intent_spam_other_trainable_capped.csv",
    index=False,
)

train_df.to_csv(OUT_DIR / "train.csv", index=False)
val_df.to_csv(OUT_DIR / "val.csv", index=False)
test_df.to_csv(OUT_DIR / "test.csv", index=False)

print("Saved:")
print(OUT_DIR / "concierge_intent_spam_other_trainable_capped.csv")
print(OUT_DIR / "train.csv")
print(OUT_DIR / "val.csv")
print(OUT_DIR / "test.csv")

Saved:
/content/concierge_classifier_data/concierge_intent_spam_other_trainable_capped.csv
/content/concierge_classifier_data/train.csv
/content/concierge_classifier_data/val.csv
/content/concierge_classifier_data/test.csv


In [38]:
# Cell 22 — Create SHA-256 hashes for model card

def sha256_file(path):
    path = Path(path)
    h = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)

    return h.hexdigest()


files_to_hash = [
    OUT_DIR / "bitext_mapped_to_concierge_labels.csv",
    OUT_DIR / "sms_spam_mapped_to_concierge_labels.csv",
    OUT_DIR / "clinc_oos_mapped_to_other.csv",
    OUT_DIR / "concierge_intent_spam_other_full_debug.csv",
    OUT_DIR / "concierge_intent_spam_other_capped_full_debug.csv",
    OUT_DIR / "concierge_intent_spam_other_trainable_capped.csv",
    OUT_DIR / "train.csv",
    OUT_DIR / "val.csv",
    OUT_DIR / "test.csv",
]

print("Dataset file hashes:\n")

for file_path in files_to_hash:
    if file_path.exists():
        print(f"{file_path.name}: {sha256_file(file_path)}")
    else:
        print(f"{file_path.name}: NOT FOUND")

Dataset file hashes:

bitext_mapped_to_concierge_labels.csv: 026dd4fac2092c5d36ef5dfb27a87cc1438fa862a8735e882672fcae02e1c6d9
sms_spam_mapped_to_concierge_labels.csv: 6b565bd21de66851290b34e9a0bcdb76007bf6f795c46ca46c2c8797c30cedec
clinc_oos_mapped_to_other.csv: 9831d46cc8656e8bf5a74c436c7ca2e3b2352d1bc4634f2b2b9550b9df95c9c6
concierge_intent_spam_other_full_debug.csv: f68eabbd428f60a88982a1c6c70cf8124e21341e9eecaf13475cb473cc089f9c
concierge_intent_spam_other_capped_full_debug.csv: dbe5cc07169c4ad97e147299e6a732866d5b2095e77701dfc87a1e78c6318206
concierge_intent_spam_other_trainable_capped.csv: 8c491fcd97600926f318fd9a58ad2f2c2ef24daf8579fff240abfd61a1a8bd1f
train.csv: 52c68f2576cdb867fef1d19d778c50a19d8009be99a8ee0f8d2f55f74de917b8
val.csv: 376775125c32c3789f3445189895492aa97ca861f14e84cf3efd967bd36a8356
test.csv: ef857cd3f1b5b65f4a3cf9be4885ee79954f687744209546067878d20c47d13d


In [39]:
# Cell 23 — Zip and download all dataset files

from google.colab import files

zip_path = "/content/concierge_classifier_data.zip"

!zip -r /content/concierge_classifier_data.zip /content/concierge_classifier_data

files.download(zip_path)

  adding: content/concierge_classifier_data/ (stored 0%)
  adding: content/concierge_classifier_data/clinc_oos_mapped_to_other.csv (deflated 78%)
  adding: content/concierge_classifier_data/bitext_mapped_to_concierge_labels.csv (deflated 93%)
  adding: content/concierge_classifier_data/concierge_intent_spam_other_trainable_capped.csv (deflated 77%)
  adding: content/concierge_classifier_data/val.csv (deflated 75%)
  adding: content/concierge_classifier_data/train.csv (deflated 76%)
  adding: content/concierge_classifier_data/concierge_intent_spam_other_capped_full_debug.csv (deflated 83%)
  adding: content/concierge_classifier_data/sms_spam_mapped_to_concierge_labels.csv (deflated 71%)
  adding: content/concierge_classifier_data/test.csv (deflated 75%)
  adding: content/concierge_classifier_data/concierge_intent_spam_other_full_debug.csv (deflated 84%)
  adding: content/concierge_classifier_data/concierge_intent_spam_other_trainable_full.csv (deflated 79%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>